# 02 Preprocessing and Feature Engineering

This notebook prepares the customer-level feature table for future segmentation. It is the main explanation layer for the preprocessing choices, so the pipeline is shown step by step instead of hidden inside one function call.

No clustering is performed and no final output file is created.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def has_raw_datasets(candidate):
    root_layout = (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists()
    project_files_layout = (candidate / "Project files" / "customer_info.csv").exists() and (candidate / "Project files" / "customer_basket.csv").exists()
    return root_layout or project_files_layout


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if has_raw_datasets(candidate):
            return candidate
    raise FileNotFoundError("Could not locate raw datasets at the repository root or in Project files/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE_DATE = pd.Timestamp.today().normalize()
print(f"Project root: {PROJECT_ROOT}")
print(f"Reference date for age and tenure: {REFERENCE_DATE.date()}")

In [ ]:
from src.data_loading import load_datasets
from src.data_audit import missing_values
from src.features import (
    RAW_NON_MODEL_COLUMNS,
    compute_basket_features,
    compute_family_features,
    compute_spend_shares_by_category,
    compute_total_lifetime_spend,
    merge_basket_features,
    parse_basket_goods,
)
from src.preprocessing import (
    drop_raw_identifier_columns,
    handle_missing_numeric_values,
    preprocess_customer_info,
)

## Pipeline Overview

The pipeline is intentionally simple:

1. Load the two raw CSV files.
2. Preprocess `customer_info` because it contains the full customer base.
3. Remove raw identifiers and handle missing numeric values.
4. Create spend features.
5. Create family features.
6. Parse and aggregate basket features by customer.
7. Left-join basket features onto the full customer table.
8. Validate that no customer was lost.

The final result should stay at 33,038 rows, one row per `customer_id` from `customer_info`.

## 1. Load Raw Data

`customer_info` is the authoritative customer table. `customer_basket` adds sampled behavior, but it does not contain every customer, so it cannot be the base table.

In [ ]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

customers_before = customer_info["customer_id"].nunique()
print(f"customer_info rows: {len(customer_info):,}")
print(f"unique customers before feature engineering: {customers_before:,}")
print(f"customer_basket rows: {len(customer_basket):,}")

## 2. Preprocess Customer Data

This step converts raw customer fields into modeling-friendly numeric fields. Birthdate becomes age, first transaction year becomes tenure, the loyalty card number becomes a yes/no flag, suspicious promotion percentages are clipped and flagged, and gender becomes indicator columns.

Raw identifiers are then removed before modeling features are created.

In [ ]:
customer_features = preprocess_customer_info(customer_info, reference_date=REFERENCE_DATE)
customer_features = drop_raw_identifier_columns(customer_features, RAW_NON_MODEL_COLUMNS)

numeric_columns = [
    column
    for column in customer_features.select_dtypes(include=[np.number]).columns
    if column != "customer_id"
]
customer_features, imputation_values = handle_missing_numeric_values(
    customer_features,
    columns=numeric_columns,
    add_indicator=True,
)

print(f"customer feature table after preprocessing: {customer_features.shape}")
print(f"numeric columns imputed: {len(imputation_values)}")
print(f"missing values after customer preprocessing: {int(customer_features.isna().sum().sum()):,}")

display(customer_features.head())

## 3. Create Spend Features

The raw spend columns are useful, but two extra views are easier to explain later: total lifetime spend captures overall value, and spend shares show each customer's product-category mix.

In [ ]:
customer_features = compute_total_lifetime_spend(customer_features)
customer_features = compute_spend_shares_by_category(customer_features)

spend_feature_preview = [
    "total_lifetime_spend",
    "spend_share_groceries",
    "spend_share_electronics",
    "spend_share_meat",
    "spend_share_fish",
    "spend_share_videogames",
]

display(customer_features[spend_feature_preview].describe().T)

## 4. Create Family Features

`kids_home` and `teens_home` are kept, and the pipeline adds simple household summaries. These help interpret family-oriented segments without forcing the model to infer them from two separate columns only.

In [ ]:
customer_features = compute_family_features(customer_features)

family_features = [
    "kids_home",
    "teens_home",
    "total_children_home",
    "has_kids_home",
    "has_teens_home",
    "has_children_home",
]

display(customer_features[family_features].describe().T)

## 5. Create Basket Features

Basket rows are parsed and summarized at customer level. These features are optional enrichments because 4,911 customers do not appear in the sampled basket file.

In [ ]:
parsed_basket, parse_errors = parse_basket_goods(customer_basket)
basket_features = compute_basket_features(parsed_basket)

print(f"parsed basket rows: {len(parsed_basket):,}")
print(f"basket parse errors: {len(parse_errors):,}")
print(f"customers with sampled baskets: {len(basket_features):,}")

display(parse_errors.head() if not parse_errors.empty else pd.DataFrame({"message": ["No parse errors found."]}))
display(basket_features.head())

## 6. Merge Basket Features Onto The Full Customer Base

The merge is a left join from `customer_info`-based features to basket features. This is the key row-preservation step: customers without sampled baskets stay in the table and receive zero basket defaults plus `has_sampled_basket = 0`.

In [ ]:
feature_table = merge_basket_features(customer_features, basket_features)
feature_table = feature_table.sort_values("customer_id").reset_index(drop=True)

customers_after = feature_table["customer_id"].nunique()
customers_without_baskets = int((feature_table["basket_count"] == 0).sum())

metadata = {
    "input_customer_count": customers_before,
    "output_customer_count": customers_after,
    "feature_table_shape": feature_table.shape,
    "basket_parse_errors": len(parse_errors),
    "customers_without_baskets": customers_without_baskets,
    "basket_features_shape": basket_features.shape,
}

print(f"feature table shape: {feature_table.shape}")
print(f"unique customers after feature engineering: {customers_after:,}")
print(f"customers without sampled baskets: {customers_without_baskets:,}")
display(feature_table.head())

## 7. Validate Customer Preservation

These checks are the guardrails for the phase. If any assertion fails, the feature table is not safe to use for clustering because it may have lost or duplicated customers.

In [ ]:
assert len(feature_table) == len(customer_info), "Feature table row count changed."
assert feature_table["customer_id"].is_unique, "customer_id is not unique after feature engineering."
assert set(feature_table["customer_id"]) == set(customer_info["customer_id"]), "Customer IDs do not match customer_info."
assert len(parse_errors) == 0, "Some basket rows could not be parsed."

no_basket = feature_table["basket_count"] == 0
assert (feature_table.loc[no_basket, "avg_basket_size"] == 0).all(), "No-basket customers must have avg_basket_size = 0."
assert (feature_table.loc[no_basket, "unique_basket_products"] == 0).all(), "No-basket customers must have unique_basket_products = 0."

print("PASS: every customer from customer_info is present exactly once.")
print(f"Customers before: {customers_before:,}")
print(f"Customers after: {customers_after:,}")
print(f"Customers without baskets retained: {int(no_basket.sum()):,}")

## Missing Values After Preprocessing

The feature table should be complete before clustering experiments. Missing raw values are handled through median imputation, and missingness indicators preserve where imputation happened.

In [ ]:
post_missing = missing_values(feature_table)
display(post_missing if not post_missing.empty else pd.DataFrame({"message": ["No missing values remain after preprocessing."]}))
print(f"Total missing values after preprocessing: {int(feature_table.isna().sum().sum()):,}")

## Engineered Feature Distributions

These summaries are not model results. They are quick checks to understand the scale and range of the main engineered fields before choosing a scaling strategy.

In [ ]:
selected_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "total_children_home",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

distribution = feature_table[selected_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
display(distribution)

## Key Feature Groups Created

This groups the feature table into concepts that are easier to discuss in a defense: demographics, household, spending, loyalty, promotion, and baskets.

In [ ]:
feature_groups = {
    "demographic": ["customer_age", "gender_female", "gender_male", "gender_unknown"],
    "family": ["kids_home", "teens_home", "total_children_home", "has_children_home"],
    "spend": [column for column in feature_table.columns if column.startswith("spend_share_")] + ["total_lifetime_spend"],
    "loyalty_complaints": ["has_loyalty_card", "loyalty_card_missing", "number_complaints"],
    "promotion": ["promotion_pct_clean", "promotion_pct_suspicious", "promotion_pct_missing"],
    "basket": ["basket_count", "avg_basket_size", "unique_basket_products", "has_sampled_basket"],
}

for group, columns in feature_groups.items():
    present = [column for column in columns if column in feature_table.columns]
    print(f"{group}: {len(present)} features")
    print(present)
    print()

## Customers With Baskets vs Without Baskets

This comparison checks whether the no-basket group looks broadly similar on customer-level fields and confirms that basket features are zero only where `has_sampled_basket = 0`.

In [ ]:
comparison_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "has_loyalty_card",
    "number_complaints",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

basket_comparison = (
    feature_table.assign(basket_status=feature_table["has_sampled_basket"].map({1: "with baskets", 0: "without baskets"}))
    .groupby("basket_status")[comparison_features]
    .agg(["count", "mean", "median"])
)
display(basket_comparison)

## Data Quality Flags To Carry Forward

These flags are not errors by themselves. They document where the raw data needed parsing, clipping, invalid-value handling, or imputation.

In [ ]:
quality_flag_columns = [
    column
    for column in feature_table.columns
    if column.endswith("_was_missing")
    or column.endswith("_missing")
    or column.endswith("_missing_or_invalid")
    or column.endswith("_suspicious")
    or column.endswith("_parse_failed")
]

quality_flags = (
    feature_table[quality_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("flagged_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(quality_flags[quality_flags["flagged_rows"] > 0])

## Phase Summary

This is the handoff point to clustering. The table is complete and row-preserving, but the modeling feature subset and scaling strategy still need to be chosen.

In [ ]:
summary = {
    "customers_before": customers_before,
    "customers_after": customers_after,
    "feature_columns_including_customer_id": feature_table.shape[1],
    "customers_without_baskets": int((feature_table["basket_count"] == 0).sum()),
    "total_missing_after_preprocessing": int(feature_table.isna().sum().sum()),
    "clustering_performed": False,
}
display(pd.DataFrame([summary]))
print("Feature engineering complete. No clustering was performed and no final output file was created.")

## What I Need To Understand For The Defense

- `customer_info` is the base because it is the only file with the full customer universe. The final assignment requires every customer to receive a cluster.
- Basket features are optional enrichments because `customer_basket` is sampled and does not include every customer.
- Customers without sampled baskets are retained so the final customer population is not reduced. Their basket aggregates are zero, and `has_sampled_basket` explains why.
- Identifiers are excluded from modeling because names, loyalty-card numbers, raw dates, and IDs are not behavioral dimensions. `customer_id` remains only as a key.
- Clustering has not been done yet because feature selection, scaling, and treatment of quality flags should be reviewed first. This notebook prepares the table; the next phase should design and compare clustering experiments.